In [1]:
import numpy as np
import time

In [ ]:
# Generate random input data
def randomInitNat(num_items: int, H: int = 2**24, seed: int = 12345) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.integers(0, H, size=num_items, dtype=np.uint32)

# Build even offsets for segmenting data
def buildEvenOffsets(N: int, M: int) -> np.ndarray:
    base, rem = divmod(N, M)
    lens = np.full(M, base, dtype=np.int64)
    lens[:rem] += 1
    offsets = np.empty(M + 1, dtype=np.int64)
    offsets[0] = 0
    np.cumsum(lens, out=offsets[1:])
    return offsets

# Segmented sort using NumPy
def segmented_sort_numpy(keys_in: np.ndarray, offsets: np.ndarray) -> np.ndarray:
    keys_out = np.empty_like(keys_in)
    # Loop over segments 
    for s, e in zip(offsets[:-1], offsets[1:]):
        # np.sort returns a sorted copy for the slice; assign into output
        keys_out[s:e] = np.sort(keys_in[s:e])
    return keys_out

In [ ]:
def sort_segmented_numpy(num_items: int, num_segments: int, runs: int = 5, seed: int = 1337) -> None:
    # Generate data
    H = 2 ** 24
    h_keys_in = randomInitNat(num_items, H=H, seed=seed)
    # Build even offsets
    h_offsets = buildEvenOffsets(num_items, num_segments)

    # Warm-up
    _ = segmented_sort_numpy(h_keys_in, h_offsets)

    # Run time 
    t0 = time.perf_counter()
    for _ in range(runs):
        _ = segmented_sort_numpy(h_keys_in, h_offsets)
    t1 = time.perf_counter()

    avg_us = (t1 - t0) * 1e6 / runs

    # Throughput in GB/s
    total_bytes = h_keys_in.nbytes
    throughput_gbps = total_bytes / (avg_us / 1e6) / 1e9

    print(f" NumPy Segmented Sort - Items: {num_items:10d} Segments: {num_segments:6d} Time: {avg_us:10.3f} µs Avg")
    print(f" NumPy Segmented Sort - Throughput: {throughput_gbps:10.3f} GB/s")


In [5]:
if __name__ == "__main__":
    sort_segmented_numpy(32000000, 32)
    sort_segmented_numpy(32000000, 320)
    sort_segmented_numpy(32000000, 3200)
    sort_segmented_numpy(32000000, 32000)
    sort_segmented_numpy(32000000, 320000)
    sort_segmented_numpy(32000000, 3200000)
    sort_segmented_numpy(32000000, 32000000)

 NumPy Segmented Sort - Items:   32000000 Segments:     32 Time: 1631807.642 µs Avg
 NumPy Segmented Sort - Throughput:      0.078 GB/s
 NumPy Segmented Sort - Items:   32000000 Segments:    320 Time: 1413253.858 µs Avg
 NumPy Segmented Sort - Throughput:      0.091 GB/s
 NumPy Segmented Sort - Items:   32000000 Segments:   3200 Time: 1058982.175 µs Avg
 NumPy Segmented Sort - Throughput:      0.121 GB/s
 NumPy Segmented Sort - Items:   32000000 Segments:  32000 Time: 834609.800 µs Avg
 NumPy Segmented Sort - Throughput:      0.153 GB/s
 NumPy Segmented Sort - Items:   32000000 Segments: 320000 Time: 784347.400 µs Avg
 NumPy Segmented Sort - Throughput:      0.163 GB/s
 NumPy Segmented Sort - Items:   32000000 Segments: 3200000 Time: 2651947.883 µs Avg
 NumPy Segmented Sort - Throughput:      0.048 GB/s
 NumPy Segmented Sort - Items:   32000000 Segments: 32000000 Time: 22104409.025 µs Avg
 NumPy Segmented Sort - Throughput:      0.006 GB/s
